In [11]:
"""ACCESS-C demo/test."""

%load_ext autoreload
%autoreload 2
import shutil
import xarray as xr

import thuner.data as data
import thuner.option as option
import thuner.analyze as analyze
import thuner.parallel as parallel
import thuner.visualize as visualize
import thuner.default as default
import thuner.config as config
import thuner.utils as utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = config.get_outputs_directory()
output_parent = base_local / f"runs/access_c/access_c_demo"
options_directory = output_parent / "options"
visualize_directory = output_parent / "visualize"

In [13]:
# Delete the output directory for the run if it already exists
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)

In [14]:
# Download the demo data
remote_directory = "s3://thuner-storage/THUNER_output/input_data/raw/ops_aps3/"
data.get_demo_data(base_local, remote_directory)

2026-06-03 19:51:59,109 - thuner.data._utils - INFO - Syncing directory /home/ewan/THUNER_output/input_data/raw/ops_aps3. Please wait.


In [15]:
# Create the dataset options
# For model datasets we generally need to specify which model run we want, in
# addition to the start and end times. Typically we want to discard spin up times.
run_start = "2021-12-01T12:00:00"  # The start time of the run we want
start = "2021-12-02T06:00:00"  # The start time of the data we want to analyze.
end = "2021-12-02T12:00:00"  # The end time of the data we want to analyze.
times_dict = {"start": start, "end": end, "run_start": run_start}

access_1km_options = data.access.AccessCOptions(
    **times_dict, name="access_1km", filename="radar_refl_1km.nc"
)
# access_maxcol shares the same native ACCESS-C grid as access_1km, so it reuses the
# regridder weights built for access_1km rather than building (and storing) its own.
access_max_col_options = data.access.AccessCOptions(
    **times_dict,
    name="access_maxcol",
    filename="maxcol_refl.nc",
    regridder_from="access_1km",
)

2026-06-03 19:51:59,567 - thuner.data.access - INFO - Generating ACCESS-C filepaths.
2026-06-03 19:51:59,568 - thuner.data.access - INFO - Generating ACCESS-C filepaths.


In [16]:
datasets=[access_1km_options, access_max_col_options]
data_options = option.data.DataOptions(datasets=datasets)
data_options.to_json(options_directory / "data.json")

grid_options = option.grid.GridOptions()
grid_options.to_json(options_directory / "grid.json")

track_options = default.track.access_c_track()
track_options.to_json(options_directory / "track.json")

2026-06-03 19:51:59,596 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.
2026-06-03 19:51:59,597 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [17]:
times = utils.generate_dataset_times(data_options.dataset_by_name("access_1km"))
parallel.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    output_directory=output_parent,
    dataset_name='access_1km',
    num_processes=4,
)

2026-06-03 19:52:00,048 - thuner.parallel - INFO - Beginning parallel tracking with 4 processes.
2026-06-03 19:52:00,057 - thuner.parallel - INFO - Pre-computing regridder weights for access_1km.
2026-06-03 19:52:00,058 - thuner.data.access - INFO - Converting access_1km dataset for time 2021-12-01T12:00:00.
2026-06-03 19:52:00,065 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 19:52:00,067 - thuner.data._utils - INFO - Building regridder; this can take a while for large grids.


2026-06-03 19:52:08,673 - thuner.utils - INFO - Grid options not set. Inferring from dataset.
2026-06-03 19:52:09,083 - thuner.parallel - INFO - Verifying existing regridder weights for access_maxcol.
2026-06-03 19:52:09,084 - thuner.data.access - INFO - Converting access_maxcol dataset for time 2021-12-01T12:00:00.
2026-06-03 19:52:09,089 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 19:52:09,090 - thuner.data._utils - INFO - Loading regridder weights from file.
2026-06-03 19:52:11,057 - thuner.utils - INFO - Grid options not set. Inferring from dataset.
2026-06-03 19:52:13,002 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/access_c/access_c_demo/interval_0.
2026-06-03 19:52:13,456 - thuner.track.track - INFO - Processing 2021-12-02T06:00:00.
2026-06-03 19:52:13,456 - thuner.utils - INFO - Updating access_1km input record for 2021-12-02T06:00:00.
2026-06-03 19:52:13,456 - thuner.data.a

In [18]:
analysis_options = analyze.mcs.AnalysisOptions()
analysis_options.to_json(options_directory / "analysis.json")
analyze.mcs.process_velocities(output_parent, profile_dataset=None)
analyze.mcs.quality_control(output_parent, analysis_options)

2026-06-03 19:52:28,604 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.
2026-06-03 19:52:28,651 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.


In [ ]:
style = "presentation"
attribute_handlers = default.visualize.grouped_attribute_handlers(output_parent, style)
figure_options = option.visualize.GroupedHorizontalAttributeOptions(
    name="mcs_attributes",
    object_name="mcs",
    style=style,
    attribute_handlers=attribute_handlers,
    altitude_titles=False,
)
visualize.attribute.series(
    output_directory=output_parent,
    start_time=start,
    end_time=end,
    figure_options=figure_options,
    dataset_name="access_1km",
    parallel_figure=True,
    by_date=False,
    num_processes=4,
)

2026-06-03 19:52:28,836 - thuner.option.grid - WARNING - shape not specified. Will attempt to infer from input.
2026-06-03 19:52:28,842 - thuner.visualize.attribute - INFO - Visualizing attributes at time 2021-12-02T06:00:00.000000000.
2026-06-03 19:52:28,899 - thuner.data.access - INFO - Converting access_1km dataset for time 2021-12-02T06:00:00.
2026-06-03 19:52:28,905 - thuner.grid - INFO - Creating new geographic grid with spacing 0.025, 0.025.
2026-06-03 19:52:28,906 - thuner.data._utils - INFO - Loading regridder weights from file.
2026-06-03 19:52:30,238 - thuner.utils - INFO - Grid options not set. Inferring from dataset.
2026-06-03 19:52:30,646 - thuner.data.access - INFO - Converting access_maxcol dataset for time 2021-12-02T06:00:00.
2026-06-03 19:52:30,652 - thuner.data._utils - INFO - Loading regridder weights from file.
2026-06-03 19:52:34,061 - thuner.visualize.attribute - INFO - Saving mcs_attributes figure for 2021-12-02T06:00:00.000000000.
2026-06-03 19:52:37,688 - th

![MCS detection and matching for ACCESS-C data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/mcs_access_c.gif)